# OpenAzrael — Retail Real-Geometry Map Pipeline

**Source-only. No AI/generated geometry.** This notebook reads the original GOG-restored `AZT.BIN`/`AZT.CUE`, extracts the CD filesystem, finds original XAF/AZF assets, exports XAF → GLB using OpenAzrael's existing exporter, renders top-down X/Z wireframes, parses `MAPS.DAT` for exact room connections, and marks only evidence-backed alternate-state room records.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!apt-get update -qq
!apt-get install -y -qq bchunk p7zip-full git
!pip -q install numpy pillow trimesh networkx matplotlib


In [ ]:
from pathlib import Path
import hashlib, json, os, re, shutil, subprocess, sys, math
import numpy as np
from PIL import Image, ImageDraw
import trimesh

DRIVE=Path('/content/drive/MyDrive/OpenAzrael')
BIN=DRIVE/'retail_extracted'/'AZT.BIN'
CUE=DRIVE/'retail_extracted'/'AZT.CUE'
GAME=DRIVE/'retail_extracted'/'GAME'/'AZRAEL'
OUT=DRIVE/'retail_geometry'
CD=OUT/'cd_extracted'
TEXTURES=OUT/'textures_png'
GLB=OUT/'glb'
ATLAS=OUT/'atlas'
META=OUT/'metadata'
for p in (OUT,CD,TEXTURES,GLB,ATLAS,META): p.mkdir(parents=True,exist_ok=True)

EXPECTED_BIN_SIZE=511_023_744
EXPECTED_BIN_MD5='42911c0462049cee17c0bbef17b24f69'

def digest(path,algo='sha256',chunk=4*1024*1024):
    h=hashlib.new(algo)
    with path.open('rb') as f:
        for b in iter(lambda:f.read(chunk),b''): h.update(b)
    return h.hexdigest()

assert BIN.is_file() and CUE.is_file(), 'AZT.BIN/AZT.CUE missing'
assert BIN.stat().st_size==EXPECTED_BIN_SIZE,(BIN.stat().st_size,EXPECTED_BIN_SIZE)
bin_md5=digest(BIN,'md5')
assert bin_md5==EXPECTED_BIN_MD5,(bin_md5,EXPECTED_BIN_MD5)
print('SOURCE VERIFIED',BIN.stat().st_size,bin_md5)

# Convert raw MODE1/2352 BIN+CUE to ISO with bchunk. Persistent and resume-safe.
ISO=OUT/'AZT01.iso'
if not ISO.exists():
    subprocess.run(['bchunk',str(BIN),str(CUE),str(OUT/'AZT')],check=True)
assert ISO.exists(), 'bchunk did not create AZT01.iso'
print('ISO',ISO.stat().st_size)

marker=CD/'.extract_complete'
if not marker.exists():
    subprocess.run(['7z','x','-y',str(ISO),f'-o{CD}'],check=True)
    marker.write_text('ok\n')
print('CD extracted',CD)

all_files=[p for p in CD.rglob('*') if p.is_file()]
xaf=sorted(p for p in all_files if p.suffix.upper()=='.XAF')
azf=sorted(p for p in all_files if p.suffix.upper()=='.AZF')
print('CD files',len(all_files),'XAF',len(xaf),'AZF',len(azf))
(META/'cd_inventory.json').write_text(json.dumps({
 'bin_md5':bin_md5,'iso_bytes':ISO.stat().st_size,'file_count':len(all_files),
 'xaf_count':len(xaf),'azf_count':len(azf),
 'xaf':[str(p.relative_to(CD)) for p in xaf],
 'azf':[str(p.relative_to(CD)) for p in azf]},indent=2))

# Clone source-safe OpenAzrael tooling.
REPO=Path('/content/OpenAzrael')
if not REPO.exists():
    subprocess.run(['git','clone','--depth','1','--branch','integration/kg001-candidate3-pppv-v2',
                    'https://github.com/semishsvk/OpenAzrael.git',str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin','integration/kg001-candidate3-pppv-v2','--depth','1'],check=True)
    subprocess.run(['git','-C',str(REPO),'checkout','integration/kg001-candidate3-pppv-v2'],check=True)
    subprocess.run(['git','-C',str(REPO),'reset','--hard','origin/integration/kg001-candidate3-pppv-v2'],check=True)
sys.path[:0]=[str(REPO),str(REPO/'tools'/'texture_pipeline')]
from azf_decode_v14 import save_png
from export_xaf_glb_v8 import parse as parse_xaf

# Decode original AZF textures. No enhancement.
azf_by_stem={}
duplicate_texture_conflicts=[]
for p in azf:
    key=p.stem.upper(); sha=digest(p)
    if key in azf_by_stem and azf_by_stem[key][1]!=sha:
        duplicate_texture_conflicts.append({'stem':key,'a':str(azf_by_stem[key][0]),'b':str(p)})
        continue
    azf_by_stem[key]=(p,sha)
for i,(key,(p,sha)) in enumerate(sorted(azf_by_stem.items()),1):
    dst=TEXTURES/(p.stem+'.png')
    if not dst.exists(): save_png(p,dst)
    if i%100==0 or i==len(azf_by_stem): print('textures',i,'/',len(azf_by_stem))

# Export every original XAF. Keep provenance, never infer room identity from number.
export_rows=[]; seen={}
for i,p in enumerate(xaf,1):
    rel=p.relative_to(CD); key=p.stem.upper()
    outstem=p.stem
    if key in seen:
        outstem=re.sub(r'[^A-Za-z0-9_.-]+','_',str(rel.with_suffix('')))
    seen[key]=seen.get(key,0)+1
    dst=GLB/(outstem+'.glb')
    try:
        scene,objs,prims,mats,resolved,missing=parse_xaf(p,TEXTURES)
        scene.export(dst)
        export_rows.append({
          'source':str(rel),'source_sha256':digest(p),'source_stem':p.stem,
          'glb':dst.name,'glb_sha256':digest(dst),'objects':len(objs),
          'primitives':len(prims),'triangles':sum(int(x.get('triangles',0)) for x in prims),
          'missing_textures':missing,'status':'PASS' if not missing else 'PASS_GEOMETRY_TEXTURE_WARN'})
    except Exception as e:
        export_rows.append({'source':str(rel),'source_stem':p.stem,'status':'FAIL','error':repr(e)})
    if i%25==0 or i==len(xaf): print('XAF',i,'/',len(xaf))
(META/'xaf_export_manifest.json').write_text(json.dumps({
 'duplicate_texture_conflicts':duplicate_texture_conflicts,'exports':export_rows},indent=2))
print('GLB PASS',sum(r.get('status','').startswith('PASS') for r in export_rows),'FAIL',sum(r.get('status')=='FAIL' for r in export_rows))

# Parse exact retail map topology.
maps=(GAME/'MAPS.DAT').read_bytes().replace(b'\x1a',b'').decode('latin1','replace')
rooms=[]; cur=None
for raw in maps.splitlines():
    line=raw.strip()
    if not line or line.startswith('//'): continue
    m=re.match(r'^ROOM\s+(\d+)\s*(.*)$',line,re.I)
    if m:
        cur={'room_id':int(m.group(1)),'name':m.group(2).strip(),'entries':[]}; rooms.append(cur); continue
    if re.match(r'^END\b',line,re.I): cur=None; continue
    if cur:
        m=re.match(r'^(\S+)\s+(-?\d+)\s+(-?\d+)(?:\s+(-?\d+))?',line)
        if m: cur['entries'].append({'asset':m.group(1),'x':int(m.group(2)),'y':int(m.group(3)),
                                     'target_room_id':int(m.group(4)) if m.group(4) else None})
edges=[{'from':r['room_id'],'to':e['target_room_id'],'asset':e['asset'],'x':e['x'],'y':e['y']}
       for r in rooms for e in r['entries'][1:] if e['target_room_id'] is not None]
(META/'retail_topology_maps_dat.json').write_text(json.dumps({
 'schema':'openazrael.retail_maps_dat_topology','room_count':len(rooms),'rooms':rooms,'edges':edges},indent=2))
print('MAPS.DAT',len(rooms),'rooms',len(edges),'directed edges')

# Bind only by exact source asset stem. No R-number guessing.
glb_by_stem={}
for r in export_rows:
    if r.get('status','').startswith('PASS'): glb_by_stem.setdefault(r['source_stem'].upper(),[]).append(r)
bindings=[]
for room in rooms:
    if not room['entries']: continue
    stem=room['entries'][0]['asset'].upper()
    matches=glb_by_stem.get(stem,[])
    bindings.append({'room_id':room['room_id'],'room_name':room['name'],'maps_asset':room['entries'][0]['asset'],
      'matches':[x['glb'] for x in matches],
      'status':'BOUND_EXACT_ASSET_STEM' if len(matches)==1 else ('AMBIGUOUS_EXACT_STEM' if len(matches)>1 else 'UNBOUND')})
(META/'room_geometry_bindings_exact.json').write_text(json.dumps(bindings,indent=2))
print('exact bindings',sum(x['status']=='BOUND_EXACT_ASSET_STEM' for x in bindings),'/',len(bindings))

# Explicit state records visible directly in MAPS.DAT.
state_groups=[
 {'state_group':'ziggurat','rooms':[43,69],'labels':['ZIGGURAT','FLOODED ZIGGURAT']},
 {'state_group':'execution_room','rooms':[36,77],'labels':['EXECUTION ROOM DRY','FLOODED EXECUTION ROOM']}]
(META/'explicit_state_groups.json').write_text(json.dumps(state_groups,indent=2))

# Render real X/Z wireframes from GLB meshes.
SKIP=re.compile(r'^(zx|zy|zq)|(^object\d*)|view|spaview|entcori',re.I)
def records(scene):
    out=[]
    for node in scene.graph.nodes_geometry:
        T,gn=scene.graph[node]
        if SKIP.search(f'{node} {gn}'): continue
        g=scene.geometry[gn]
        v=trimesh.transform_points(g.vertices,T)
        f=np.asarray(g.faces,dtype=np.int64)
        if len(v) and len(f): out.append((v,f))
    return out
def render_wire(path,out,w=600,h=420,pad=18):
    s=trimesh.load(path,force='scene'); rec=records(s)
    if not rec:return None
    aa=np.vstack([v[:,[0,2]] for v,_ in rec]); mn=aa.min(0); mx=aa.max(0)
    span=np.maximum(mx-mn,1e-9); scale=min((w-2*pad)/span[0],(h-2*pad)/span[1]); cen=(mn+mx)*.5
    def tx(p):
        q=(p-cen)*scale; return float(q[0]+w/2),float(h/2-q[1])
    im=Image.new('RGB',(w,h),'white'); d=ImageDraw.Draw(im); ec=0
    for v3,faces in rec:
        v=v3[:,[0,2]]; es=set()
        for f in faces:
            for a,b in ((f[0],f[1]),(f[1],f[2]),(f[2],f[0])):
                a,b=int(a),int(b)
                if a>b:a,b=b,a
                es.add((a,b))
        ec+=len(es)
        for a,b in es:d.line([tx(v[a]),tx(v[b])],fill=(20,20,20),width=1)
    im.save(out)
    return {'xz_bounds':[mn.tolist(),mx.tolist()],'projected_edges':ec}

wire=[]
passed=[r for r in export_rows if r.get('status','').startswith('PASS')]
for i,r in enumerate(passed,1):
    gp=GLB/r['glb']; pp=ATLAS/(gp.stem+'_topdown.png')
    try:
        info=render_wire(gp,pp)
        wire.append({'glb':gp.name,'preview':pp.name,**(info or {})})
    except Exception as e: wire.append({'glb':gp.name,'status':'FAIL','error':repr(e)})
    if i%25==0 or i==len(passed):print('wireframes',i,'/',len(passed))
(META/'geometry_wireframe_atlas.json').write_text(json.dumps(wire,indent=2))

# Contact sheet: every tile is a render of original XAF-derived geometry.
thumbs=[(r['glb'],Image.open(ATLAS/r['preview']).convert('RGB')) for r in wire if r.get('preview') and (ATLAS/r['preview']).exists()]
if thumbs:
    cols=4; tw,th,label=320,224,26; nr=math.ceil(len(thumbs)/cols)
    sh=Image.new('RGB',(cols*tw,nr*(th+label)),'white'); dr=ImageDraw.Draw(sh)
    for i,(name,im) in enumerate(thumbs):
        x=(i%cols)*tw; y=(i//cols)*(th+label); im.thumbnail((tw,th))
        sh.paste(im,(x+(tw-im.width)//2,y+(th-im.height)//2)); dr.text((x+5,y+th+5),name,fill='black')
    sh.save(OUT/'retail_geometry_contact_sheet.png')

# Extract all CD files whose stem exactly matches a MAPS.DAT room-map asset;
# this discovers original 2D map sprites/room resources without assuming extensions.
wanted={e['asset'].upper() for r in rooms for e in r['entries']}
map_asset_matches={}
for p in all_files:
    if p.stem.upper() in wanted:
        map_asset_matches.setdefault(p.stem.upper(),[]).append(str(p.relative_to(CD)))
(META/'maps_asset_file_matches.json').write_text(json.dumps(map_asset_matches,indent=2))

status={
 'status':'PASS' if xaf else 'BLOCKED_NO_XAF_ON_CD',
 'bin_md5':bin_md5,'cd_files':len(all_files),'xaf_found':len(xaf),'azf_found':len(azf),
 'glb_exported':sum(r.get('status','').startswith('PASS') for r in export_rows),
 'wireframes':sum('preview' in r for r in wire),
 'maps_rooms':len(rooms),'maps_directed_edges':len(edges),
 'exact_geometry_bindings':sum(x['status']=='BOUND_EXACT_ASSET_STEM' for x in bindings),
 'map_asset_stems_matched_on_cd':len(map_asset_matches),
 'explicit_state_groups':state_groups,
 'output_folder':str(OUT),
 'rule':'No geometry or room binding is invented. Unbound rooms stay UNBOUND.'}
(META/'retail_geometry_pipeline_status.json').write_text(json.dumps(status,indent=2))
print(json.dumps(status,indent=2))
print('DONE:',OUT)
